In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_3.py ──
"""
Shared infrastructure for MLFP04 Exercise 3 — Dimensionality Reduction.

Contains: data loading, scaling, common output directory, KMeans-based
silhouette evaluation in the embedding space. Technique-specific code
(PCA/KPCA/t-SNE/UMAP algorithms and their plots) lives in the per-
technique files, NOT here.

"""

from pathlib import Path

import numpy as np
import polars as pl
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# OUTPUT + REPRODUCIBILITY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "ex3_dimreduce"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEFAULT_N_CLUSTERS = 4

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — E-commerce customers (reused from MLFP03)
# ════════════════════════════════════════════════════════════════════════


def load_customer_matrix() -> tuple[np.ndarray, list[str], pl.DataFrame]:
    """Load e-commerce customers, standardise numeric features.

    Returns:
        X          : (n_samples, n_features) standardised float matrix
        feature_cols: list of feature column names in order
        df_raw     : the raw polars DataFrame before scaling
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")

    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
        and c not in ("customer_id",)
    ]

    df_clean = customers.drop_nulls(subset=feature_cols)
    X_raw, _, _ = to_sklearn_input(df_clean, feature_columns=feature_cols)

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)
    return X, feature_cols, df_clean


# ════════════════════════════════════════════════════════════════════════
# EMBEDDING-SPACE CLUSTER QUALITY
# ════════════════════════════════════════════════════════════════════════


def evaluate_embedding_silhouette(
    embedding: np.ndarray,
    n_clusters: int = DEFAULT_N_CLUSTERS,
    random_state: int = RANDOM_STATE,
) -> float:
    """Fit KMeans in the embedding space and return the silhouette score.

    This is the standard "does the reducer preserve structure?" probe used
    across all five technique files. Returns -1.0 when only one cluster is
    found (e.g. collapsed embedding).
    """
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=5)
    labels = km.fit_predict(embedding)
    if len(set(labels)) < 2:
        return -1.0
    return float(silhouette_score(embedding, labels))


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — used by KPCA / t-SNE / UMAP / Isomap for kernel-cost paths
# ════════════════════════════════════════════════════════════════════════


def subsample_indices(
    n_samples: int, n_target: int, random_state: int = RANDOM_STATE
) -> np.ndarray:
    """Deterministic subsample indices for expensive O(n^2) methods."""
    rng = np.random.default_rng(random_state)
    return rng.choice(n_samples, min(n_target, n_samples), replace=False)




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 3.2: Kernel PCA (nonlinear dim reduction)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Apply the kernel trick to extend PCA to nonlinear manifolds
#   - Compare linear, RBF, and polynomial kernels
#   - Tune the RBF gamma hyperparameter (narrow vs wide kernel)
#   - Recognise Kernel PCA's memory wall (O(n^2) kernel matrix)
#
# PREREQUISITES: 01_pca.py (understand linear PCA first).
#
# ESTIMATED TIME: ~25 min
#
# TASKS:
#   1. Theory — the kernel trick in one paragraph
#   2. Build — fit Kernel PCA with linear, RBF, polynomial kernels
#   3. Train — sweep gamma for RBF, record silhouette per config
#   4. Visualise — silhouette bar chart across kernel configurations
#   5. Apply — Grab Singapore driver-behaviour fraud screening
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

from sklearn.decomposition import KernelPCA

from kailash_ml import ModelVisualizer



## THEORY — the kernel trick


In [ ]:
# Pretend we ran PCA in a much richer feature space phi(x), without ever
# computing phi(x) explicitly:  K(x_i, x_j) = <phi(x_i), phi(x_j)>.
# Do eigen-decomposition on the n x n kernel matrix K. Kernels:
#   - linear:     K(x, y) = x . y   (equivalent to standard PCA)
#   - RBF:        K(x, y) = exp(-gamma * ||x - y||^2)
#   - polynomial: K(x, y) = (gamma * x . y + coef0)^degree
# COST: O(n^2) in memory. Subsample first.



## TASK 2 — BUILD: data + subsample for kernel cost


In [ ]:
X, feature_cols, _ = load_customer_matrix()
n_samples, n_features = X.shape
print(f"=== E-commerce customers ===  n={n_samples:,}, p={n_features}")

# TODO: build a 3,000-row subsample of X. Use the shared helper.
# Hint: idx = subsample_indices(n_samples, n_target=3000)
idx = ____
X_sub = X[idx]
print(f"Subsampled for kernel PCA: {X_sub.shape[0]:,} rows")



## TASK 3 — TRAIN: sweep kernels + gamma values


In [ ]:
kernel_configs = [
    {"kernel": "linear", "params": {}, "label": "linear"},
    {"kernel": "rbf", "params": {"gamma": 0.1}, "label": "rbf (gamma=0.1)"},
    {"kernel": "rbf", "params": {"gamma": 1.0}, "label": "rbf (gamma=1.0)"},
    {
        "kernel": "poly",
        "params": {"degree": 3, "gamma": 0.1},
        "label": "poly (deg=3)",
    },
]

kernel_results: dict[str, dict] = {}

print(f"\n=== Kernel PCA sweep ===")
print(f"{'kernel':<20}{'silhouette':>14}{'time (s)':>12}")
print("-" * 46)

for cfg in kernel_configs:
    t0 = time.time()
    # TODO: build a KernelPCA with n_components=8 and cfg['kernel'] and
    # **cfg['params']. Use random_state=42.
    # Hint: KernelPCA(n_components=8, kernel=..., random_state=42, **...)
    kpca = ____
    # TODO: fit_transform X_sub to produce the embedding.
    X_embed = ____
    elapsed = time.time() - t0

    # TODO: score the embedding using the shared silhouette helper.
    # Hint: evaluate_embedding_silhouette(X_embed)
    sil = ____
    kernel_results[cfg["label"]] = {"silhouette": sil, "time_s": elapsed}
    print(f"{cfg['label']:<20}{sil:>14.4f}{elapsed:>12.2f}")



### Checkpoint 1


In [ ]:
assert len(kernel_results) == 4, "Must evaluate all 4 kernel configurations"
linear_sil = kernel_results["linear"]["silhouette"]
print(
    "\n[ok] Checkpoint 1 — linear silhouette="
    f"{linear_sil:.4f} establishes the PCA baseline to beat"
)



## TASK 4 — VISUALISE: silhouette across kernels


In [ ]:
viz = ModelVisualizer()
fig = viz.metric_comparison(
    {label: {"Silhouette": res["silhouette"]} for label, res in kernel_results.items()}
)
fig.update_layout(title="Kernel PCA: cluster quality across kernels")
kernel_path = OUTPUT_DIR / "02_kernel_pca_silhouette.html"
fig.write_html(str(kernel_path))
print(f"\nSaved: {kernel_path}")

print("\nInterpretation:")
print("  - Linear is your baseline: if nothing beats it, use ordinary PCA.")
print("  - RBF with small gamma (wide kernel) = smooth global manifold.")
print("  - RBF with large gamma (narrow kernel) = local, more complex fit.")
print("  - Poly captures feature interactions at the cost of instability.")
print("  - Kernel PCA has no inverse_transform — you cannot reconstruct X.")



## TASK 5 — APPLY: Grab Singapore driver-behaviour fraud screening


In [ ]:
# SCENARIO: Grab's risk team screens ~50 behavioural features/driver/week
# to find fraud rings. Rings form CURVED clusters — linear PCA misses
# them. Kernel PCA with an RBF kernel surfaces the rings as separable
# blobs. Catching rings 1 week earlier saves ~S$45K per ring, times ~30
# rings/yr = S$1.35M/yr in avoided payout leakage. Downside: no inverse
# transform, and the kernel matrix is O(n^2) so we subsample per city.

best_label, best = max(kernel_results.items(), key=lambda kv: kv[1]["silhouette"])
print(f"\n=== Grab-style fraud-screening projection ===")
print(f"  Linear PCA silhouette : {linear_sil:.4f}")
print(f"  Best kernel           : {best_label}")
print(f"  Best silhouette       : {best['silhouette']:.4f}")
lift = best["silhouette"] - linear_sil
print(f"  Lift over linear PCA  : {lift:+.4f}")



## REFLECTION


[x] Applied the kernel trick to lift PCA into nonlinear feature spaces
  [x] Compared linear, RBF, polynomial kernels on the same data
  [x] Swept the RBF gamma hyperparameter (narrow vs wide)
  [x] Measured the O(n^2) kernel-matrix cost firsthand
  [x] Sized Kernel PCA for a production fraud-screening pipeline

  KEY INSIGHT: Kernel PCA gives you a curved coordinate system, but you
  pay in memory. Before reaching for the kernel trick, ask: does linear
  PCA already solve the problem?

  Next: 03_tsne.py drops linear algebra for a probabilistic model.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

